In [1]:
import pymmcore_plus
from pymmcore_plus import CMMCorePlus
from useq import MDAEvent
from useq import MDASequence
from pathlib import Path
import json
import useq
import numpy as np
import tifffile as tiff
import datetime
import stages_movement
from stages_movement import controller
from VoiceCoil_nidaqmx import DAQ
import time
mmc = None
DAQ_VC = None

In [2]:
class MDA:
 
    def __init__(
        self,
        config_path: str = r"\Users\Hannah\Desktop\configuration\PVCAM_only.cfg"
    ):
        
        # Set the first instance of this class as the global singleton
        global mmc
        if mmc is not None:
            mmc.unloadAllDevices()
        if mmc is None:
            mmc = CMMCorePlus.instance()
        mmc.enableDebugLog(True)
        # Load the correct configuration file. NB! If a new configuration file is created, change the path to the new configuration file
        mmc.loadSystemConfiguration(config_path)
        mmc.setAutoShutter(False)
        #settings needed to set up the DAQ
        global DAQ_VC
        if DAQ_VC is None:
            DAQ_VC = DAQ()
        
        
        #Setup the default Camera settings for 1 fps rolling shutter.
        self._exposure  = 2.44
        self._scan_width = 8
        self._scan_direction = 'Up'
        self._trigger  = 'Edge Trigger'
        self._Port = 'Dynamic Range'
        self._save = False
        self._setup = False
        self._controller = None
        self._sequence = None
        self._cali_path = DAQ_VC.cali_path
        self._stack_height = 1
        self._X = 1
        self._current_X = 1
        self._current_Y = 1
        self._Y = 1
        self._meta = 1
        self._channels = ['488']
        self._cameras = 1
        self._frames = 1
        self._filename = 'Zstack'
        self._foldername = 'Default'
        self._save = False
        self._z_stepsize = 0.0002
        self._silence = False
        self._tiles = 0
        self._pause_duration = 2
        self._start_pos = []
        self._tile_move = 0.608
        try:
            self._controller = controller
            self._controller.connect()
        except Exception as e:
            print(f'The controller could not connect to the stage: {e}')
        self._boundary = np.array([[-40, 20], [0.5, 18],[-0.5, 5]])
        self._home = np.array([-20, 9, 0])
        
    @property
    def save(self) -> bool:
        return getattr(self,"_save",None)
    @save.setter
    def save(self, value: bool):
        self._save = value

    @property
    def exposure(self) -> float:
        return getattr(self,"_exposure",None)
    @exposure.setter
    def exposure(self, value: float):
        self._exposure = value
    
    @property
    def scan_width(self) -> int:
        return getattr(self,"_scan_width",None)
    @scan_width.setter
    def scan_width(self, value: int):
        self._scan_width = value
        
    @property
    def cali_path(self) -> str:
        return getattr(self,"_cali_path",None)
    @cali_path.setter
    def cali_path(self, value: str):
        DAQ_VC.cali_path = value
        self._cali_path = DAQ_VC.cali_path            
    
    def connect_stages(self):
        try:
            self._controller.connect()
        except Exception as e:
            print(f'The controller could not connect to the stage: {e}')
    
    def _pause(
        self,
        duration: float = None
    ):
        if duration is not None:
            self._pause_duration = duration
        if self._current_X <= self._X:
            self._current_X += 1
            move_to_coordinates = [self._start_pos[0] + self._tile_move, self._start_pos[1], self._start_pos[2]]
        else:
            self._current_X = 1
            self._current_Y += 1
            move_to_coordinates = [self._start_pos[0], self._start_pos[1] + self._tile_move, self._start_pos[2]]
        DAQ_VC.stop()
        
        self._move_to(move_to_coordinates)
        time.sleep(self._pause_duration)
        DAQ_VC.start()
        
    
    def _saving(
        self
    ):
        while True:
            if mmc.getRemainingImageCount() > 0:
                image = mmc.popNextImage()
                self._tif.write(image, photometric='minisblack')
                print('Image saving')
                break
    def save_thread(
        self
    ):
        self._jog(self._z_stepsize)
        self._idx_frame += 1
        self._idx_slice += 1
        if self._save:
            self._saving()
        if self._idx_frame >= self._frames:
            time.sleep(0.2)
            self.stop_sequence()
            return print('Acquisition done!')
        if self._idx_slice > self._stack_height:
            self._stack += 1
            if self._save:
                self._tif.close()
                if self._stack <= self._tiles:
                    self._tif = tiff.TiffWriter(f"{self._tif_path}{self._stack:05d}.tif")
            self._idx_slice = 1
            self._pause()

    
    def _setup_tiff(
        self
    ):    
        today =datetime.datetime.now()
        self._idx_frame = 0
        self._idx_slice = 0
        self._stack = 0
        if self._foldername == 'Default':
            time_m_s = today.strftime("%H_%M")
            Path(f"{self._datestring}\\{time_m_s}").mkdir(parents=True, exist_ok=True)
            self._tif_path = f"{Path(self._datestring)}\\{time_m_s}\\{self._filename}"
        else:
            Path(f"{self._datestring}\\{self._foldername}").mkdir(parents=True, exist_ok=True)
            self._tif_path = f"{Path(self._datestring)}\\{self._foldername}\\{self._filename}"
        self._tif = tiff.TiffWriter(f"{self._tif_path}{self._stack:05d}.tif")

        
        
    def _save_path(
        self
    ):
        today =datetime.datetime.now()   # Get date
        self._datestring = today.strftime("%Y-%m-%d")  # Date to the desired string format
        Path(self._datestring).mkdir(parents=True, exist_ok=True)   # Create folder
    
    def _set_callback(
        self
    ):
        DAQ_VC.register_save(self.save_thread)
        
    def setup_save(
        self
    ):

        self._save_path()
        self._set_callback()
        if self._save is False:
            print('Saving is not enabled!')
        
        
        
    def _setup_camera(self):
        for i in range(self._cameras):
            mmc.setProperty(f"Camera-{i+1}",'Exposure',self._exposure),
            mmc.setProperty(f"Camera-{i+1}",'TriggerMode',self._trigger)
            mmc.setProperty(f"Camera-{i+1}",'ScanDirection',self._scan_direction)
            mmc.setProperty(f"Camera-{i+1}",'ScanMode','Scan Width'),
            mmc.setProperty(f"Camera-{i+1}",'ScanWidth',self._scan_width)
            mmc.setProperty(f"Camera-{i+1}",'Port',self._Port)
            
    def _calculate_frames(self):
        self._frames = self._stack_height * self._X * self._Y * len(self._channels) * self._meta * self._cameras
        self._tiles = self._X * self._Y
    
    def _setup_daq(self):
        DAQ_VC.program_waveforms(self._channels)
    
    def setup_sequence(
        self,
        z_depth: float,
        z_stepsize: float = None,
        x_tiles: int = None,
        y_tiles: int = None,
        meta: int = None,
        channels: str = None,
        cameras: int = None,
        overlap: float = 5,
        saving: bool = None,
        silence: bool = None,
        filename: str = None,
        foldername: str = None,        
        keep_save_on: bool = False
    ):
        
        if z_stepsize is not None:
            self._z_stepsize = z_stepsize
        if x_tiles is not None:
            self._X = x_tiles
        if y_tiles is not None:
            self._Y = y_tiles
        if meta is not None:
            self._meta = meta
        if channels is not None:
            self._channels = channels
        if cameras is not None:
            self._cameras = cameras
        if saving is not None:
            self._save = saving
        if silence is not None:
            self._silence = silence
        if filename is not None:
            self._filename = filename
        if foldername is not None:
            self._fodlername = foldername
        
        self._tile_move = 0.64*(100-overlap)/100
        
        self._stack_height = round(z_depth / (self._z_stepsize))
        
        self._calculate_frames()
        
        self._setup_camera()
        
        self._setup_daq()
        
        self.setup_save()

            
        

        
    def run_sequence(self):
        self._setup_tiff()
        self.enable_all()
        self._start_pos = stages_movement.get_pos(controller)
        self._current_X = 1
        self._current_Y = 1
        mmc.startSequenceAcquisition(
                                self._frames,
                                 0,
                                 True
                                )
        
        if self._trigger == 'Edge Trigger':
            DAQ_VC.start()
            
    def stop_sequence(self):
        mmc.stopSequenceAcquisition()
        if self._trigger == 'Edge Trigger':
            DAQ_VC.stop()
        try:
            if self._save:
                if mmc.getRemainingImageCount() > 0:
                    image = mmc.popNextImage()
                    self._tif.write(image, photometric='minisblack')
                    self._tif.close()
                else:
                    try:
                        self._tif.close()
                    except:
                        pass
        except Exception as e:
            print(f"could not close tiff: {e}")
        self._idx_frame = 0
        self._idx_slice = 0
        self._stack = 0
        
    def set_home(
        self
    ):
        ax0 = self._controller.axes[0]
        ax1 = self._controller.axes[1]
        ax2 = self._controller.axes[2]
        
        ax0_bounds = [-40, 2]
        ax1_bounds = np.zeros(2)
        ax2_bounds = [-0.5, 5]
        self._boundary = None
        self._home = None
        
        if ax1.enabled==True:
            ax1.disable()
        if ax0.enabled==True:
            ax0.disable()
        if ax2.enabled==False:
            ax2.enable()
        
        for i in range(2):
            input('Move the stage to one edge')
            ax1_bounds[i] = ax1.rpos
        
        self._controller.enable_all()
        
        ax1_bounds_sorted = np.sort(ax1_bounds)
        ax0_home = ax0.rpos
        ax1_home = (ax1_bounds[1] + ax1_bounds[0])/2
        ax2_home = 0
        self._home = np.array([ax0_home, ax1_home, ax2_home])
        self._boundary = np.array([ax0_bounds, ax1_bounds_sorted, ax2_bounds])
    
    def _jog(self,
            distance: float,
            axis: int = 2
            ):
        
        stages_movement.jog(
            axis,
            distance,
            self._controller,
            self._boundary
        )
    
    def _move_to(
        self,
        coordinates: np.array
    ):
        
        stages_movement.move_to(
            coordinates, 
            self._controller, 
            self._boundary
        )
    
    
    def move_home(self):
        
        stages_movement.move_to(
            self._home, 
            self._controller,
            self._boundary
        )
        
    def disable_x(self):
        ax0 = self._controller.axes[0]
        ax0.disable()
        
    def enable_x(self):
        ax0 = self._controller.axes[0]
        ax0.enable()
        
    def disable_y(self):
        ax1 = self._controller.axes[1]
        ax1.disable()
        
    def enable_y(self):
        ax1 = self._controller.axes[1]
        ax1.enable()
        
    def disable_z(self):
        ax2 = self._controller.axes[2]
        ax2.disable()
        
    def enable_z(self):
        ax2 = self._controller.axes[2]
        ax2.enable()
    
    def enable_all(self):
        ax0 = self._controller.axes[0]
        ax1 = self._controller.axes[1]
        ax2 = self._controller.axes[2]
        if not ax0.enabled:
            ax0.enable()
        if not ax1.enabled:
            ax1.enable()
        if not ax2.enabled:
            ax2.enable()
            
    def disable_all():
        ax0 = self._controller.axes[0]
        ax1 = self._controller.axes[1]
        ax2 = self._controller.axes[2]
        if ax0.enabled:
            ax0.disable()
        if ax1.enabled:
            ax1.disable()
        if ax2.enabled:
            ax2.disable()
    
    def close(self):
        mmc.unloadAllDevices()
        try:
            DAQ_VC.close()
        except Exception as e:
            print(f"Could not close DAQ: {e}")
        try:
            self._controller.disconnect()
        except Exception as e:
            print(f"Could not disconnect stage: {e}")

In [3]:
test = MDA()

In [4]:
test.save = True

In [5]:
test.move_home()

Movement has finished. from position, [-14.438144531250003, 8.069492187500002, 0.5558478895771256] to [-20.0, 9.0, 0.0]


In [6]:
test.setup_sequence(z_depth = 0.0025, x_tiles = 2, y_tiles = 2, channels = ['488'])

In [7]:
test.run_sequence()

Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Movement has finished. from position, [-20.0, 9.0, 0.0026000000000000007] to [-19.392, 9.0, 0.0]
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Movement has finished. from position, [-19.392, 9.0, 0.0024000000000000007] to [-19.392, 9.0, 0.0]
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Movement has finished. from position, [-19.392, 9.0, 0.0024000000000000007] to [-20.0, 9.608, 0.0]
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving
Image saving


In [8]:
test.stop_sequence()

In [9]:
test.move_home()

Movement has finished. from position, [-20.0, 9.608, 0.0018000000000000004] to [-20.0, 9.0, 0.0]


In [10]:
test.disable_x()
test.disable_y()
test.disable_z()

In [11]:
test.close()

In [ ]:
print(test._idx_frame)
print(test._frames)